# Tiny LLM Decoder 6 (Generation - Greedy)

In the sixth version we are focusing not on training but inference.

In [1]:
import numpy as np
import scipy.stats as stats
import torch
import torch.nn as nn
from torch.nn import functional as F

In [2]:
# 1. Check for NVIDIA CUDA (Linux / Colab)
if torch.cuda.is_available():
    device = torch.device("cuda")
# 2. Check for Apple Silicon MPS (M2 Mac)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
# 3. Fallback to CPU
else:
    device = torch.device("cpu")

print(f'Using device: {device}')

Using device: mps


## LLM Architecture

At high level, the training process is as follows:

```text

Text_processing -> Transformer Network -> Compare Loss -> Back Propagation
```

The inference process is as follows:

```text
Text_processing -> Transformer Network -> Predict Next Token -> Repeat until token size reached
```

### Corpus

In [3]:
# Text corpus
corpus = """
mary had a little lamb
its fleece was white as snow
and everywhere that mary went
the lamb was sure to go
mary had a little lamb
the lamb was sure to go
"""


### Parameters

In [4]:
# Set Parameters

# random seed
random_seed = 42
torch.manual_seed(random_seed)

# d_model is embedding dimension
d_model = 4

# block_size is the same as context window or maximum sequence length
block_size = 5



## Text Processing and Tokenization

In [5]:
# Tokenization

def tiny_tokenization(corpus_or_prompt):
  return corpus_or_prompt.split()

# Split corpus into words
word_tokens = tiny_tokenization(corpus)

# Use set to isolate unique words
vocab = sorted(list(set(word_tokens)))
vocab_size = len(vocab)

# Create token id and create lookup table from word to token id
# i is the index or tken id in this case, so we assign token id using sequential number
words_to_id_ref = {word: i for i, word in enumerate(vocab)}

# Create lookup table from id to words
id_to_words_ref = {i: word for i, word in enumerate(vocab)}

In [6]:
vocab_size

19

In [7]:
# convert corpus to id
corpus_in_id = [words_to_id_ref[word] for word in word_tokens]


## Class and Function Setup

In [8]:
def get_sequence_block(block_size, start_position, corpus_in_id):
  """
  This function extract the block in id from the corpus in id
  User need to supply the block size and the start position
  Return: X, Y, next_start_position
  """
  X = corpus_in_id[start_position: start_position+block_size]
  Y = corpus_in_id[start_position+1: start_position+block_size+1]
  return X, Y

In [9]:
def get_N(corpus_in_id, block_size):
  """
  This function calculate the number of block we need to run for the entire corpus
  """
  N = (len(corpus_in_id)-1) // block_size
  return N

In [10]:
class TinyGPT(nn.Module):

    def __init__(self, vocab_size, d_model):
        super().__init__()

        # Initialize trainable components here
        self.embeddings = nn.Embedding(vocab_size, d_model)
        self.WQ = nn.Parameter(torch.randn(d_model, d_model))
        self.WK = nn.Parameter(torch.randn(d_model, d_model))
        self.WV = nn.Parameter(torch.randn(d_model, d_model))

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model, bias=False),
            nn.ReLU(),
            nn.Linear(d_model, d_model, bias=False)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, token_ids):

        # Perform Transformer calculations here
        # Word embeddings + positional encoding
        X_input = self.embeddings(token_ids)
        position = torch.arange(
            len(token_ids),
            dtype=X_input.dtype,
            device=X_input.device
        ).reshape(-1, 1)
        X = X_input + position

        # Q, K, V projections
        Q = X @ self.WQ
        K = X @ self.WK
        V = X @ self.WV

        # Scaled dot-product attention
        attention_score = Q @ K.T
        stable_factor = self.WQ.shape[-1] ** 0.5
        attention_stable = attention_score / stable_factor

        # Causal masking + softmax
        causal_mask = torch.triu(
            torch.full_like(attention_stable, float("-inf")),
            diagonal=1
        )
        attention_stable_mask = attention_stable + causal_mask
        attention_softmax = torch.softmax(attention_stable_mask, dim=-1)

        # Attention output + residual + norm1
        attention_output = attention_softmax @ V

        # Residual Layer
        attention_output_residual = attention_output + X

        # norm1 layer
        attention_output_residual_scaled = self.norm1(attention_output_residual)

        # FFN + residual + norm2
        ffn_1 = self.ffn(attention_output_residual_scaled)

        # Residual 2 Layer
        seond_residual = ffn_1 + attention_output_residual_scaled

        # Norm 2 Layer
        transformer_output = self.norm2(seond_residual)

        # Vocabulary projection
        logits = transformer_output @ self.embeddings.weight.T

        return logits

### Training Setup

In [11]:
def llm_train(corpus_in_id, block_size, model, criterion, optimizer, epoch=10):

  # Get N
  N = get_N(corpus_in_id, block_size=block_size)

  for i in range(epoch):

    print(f"epoch {i}")

    for j in range(N):

      # Get block
      X_id, Y_id = get_sequence_block(
          block_size=block_size,
          start_position=j * block_size,
          corpus_in_id=corpus_in_id
      )
      X_tensor = torch.tensor(X_id, dtype=torch.long, device=device)
      Y_tensor = torch.tensor(Y_id, dtype=torch.long, device=device)

      optimizer.zero_grad()

      logits = model(X_tensor)
      loss = criterion(logits, Y_tensor)
      print(f"Block {j} : Loss = {loss.item():.4f}")

      loss.backward()
      optimizer.step()

### Train the Entire Corpus

In [12]:
model = TinyGPT(vocab_size=vocab_size, d_model=d_model).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [13]:
llm_train(
    corpus_in_id=corpus_in_id,
    block_size=block_size,
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    epoch=5000
)

epoch 0
Block 0 : Loss = 4.5187
Block 1 : Loss = 2.8014
Block 2 : Loss = 4.6089
Block 3 : Loss = 4.3268
Block 4 : Loss = 3.4730
Block 5 : Loss = 3.9352
epoch 1
Block 0 : Loss = 4.3720
Block 1 : Loss = 2.7273
Block 2 : Loss = 4.3967
Block 3 : Loss = 4.1483
Block 4 : Loss = 3.4424
Block 5 : Loss = 3.7677
epoch 2
Block 0 : Loss = 4.2537
Block 1 : Loss = 2.6872
Block 2 : Loss = 4.2381
Block 3 : Loss = 3.9977
Block 4 : Loss = 3.4158
Block 5 : Loss = 3.6291
epoch 3
Block 0 : Loss = 4.1522
Block 1 : Loss = 2.6658
Block 2 : Loss = 4.1127
Block 3 : Loss = 3.8658
Block 4 : Loss = 3.3902
Block 5 : Loss = 3.5108
epoch 4
Block 0 : Loss = 4.0622
Block 1 : Loss = 2.6554
Block 2 : Loss = 4.0094
Block 3 : Loss = 3.7482
Block 4 : Loss = 3.3653
Block 5 : Loss = 3.4078
epoch 5
Block 0 : Loss = 3.9807
Block 1 : Loss = 2.6519
Block 2 : Loss = 3.9219
Block 3 : Loss = 3.6423
Block 4 : Loss = 3.3410
Block 5 : Loss = 3.3169
epoch 6
Block 0 : Loss = 3.9064
Block 1 : Loss = 2.6531
Block 2 : Loss = 3.8463
Block 3 

We intentionally increase the epoch to get 100% accuracy. This is for model evaluation and inference later.

In [17]:
def evaluate_blocks(model, criterion, corpus_in_id, block_size, N):
    model.eval()

    with torch.no_grad():
        for i in range(N):
            start = i * block_size

            X_id, Y_id = get_sequence_block(
                block_size=block_size,
                start_position=start,
                corpus_in_id=corpus_in_id
            )

            X = torch.tensor(X_id, dtype=torch.long, device=device)
            Y = torch.tensor(Y_id, dtype=torch.long, device=device)

            logits = model(X)
            loss = criterion(logits, Y)

            print(f"Block {i}: Loss = {loss.item():.4f}")

            predicted_ids = torch.argmax(logits, dim=-1)
            correct = (predicted_ids == Y).sum().item()
            accuracy = correct / len(Y)

            print(f"Block {i}: Accuracy = {accuracy:.4f}")

            print("-" * 50)
            print(f"Block {i}")
            for k in range(5):
              print(f"Position {k}, expected = {id_to_words_ref[int(Y[k])]}, predicted = {id_to_words_ref[int(predicted_ids[k])]}")
            print("-" * 50)

In [18]:
N = get_N(corpus_in_id, block_size=block_size)


In [19]:
evaluate_blocks(model=model, criterion=criterion, corpus_in_id=corpus_in_id, block_size=block_size, N=N)

Block 0: Loss = 0.0057
Block 0: Accuracy = 1.0000
--------------------------------------------------
Block 0
Position 0, expected = had, predicted = had
Position 1, expected = a, predicted = a
Position 2, expected = little, predicted = little
Position 3, expected = lamb, predicted = lamb
Position 4, expected = its, predicted = its
--------------------------------------------------
Block 1: Loss = 0.0110
Block 1: Accuracy = 1.0000
--------------------------------------------------
Block 1
Position 0, expected = fleece, predicted = fleece
Position 1, expected = was, predicted = was
Position 2, expected = white, predicted = white
Position 3, expected = as, predicted = as
Position 4, expected = snow, predicted = snow
--------------------------------------------------
Block 2: Loss = 0.0119
Block 2: Accuracy = 1.0000
--------------------------------------------------
Block 2
Position 0, expected = and, predicted = and
Position 1, expected = everywhere, predicted = everywhere
Position 2, exp

## Inference

During inference, there are many decoding method:

- Greedy
- Sampleing
- Temperature Sampling
- Top K Sampling
- Top P Sampling
- Beam Search

We will be starting with greedy approach. In greedy approach, in the next token prediction logits, we convert to softmax and get the token with the highest probability.

In greedy decoding, we use argmax to get the best probabilities in the prediction.

### Inference 1

In [34]:
user_prompt = "mary"
user_prompt

'mary'

In [35]:
def prompt_processing(user_prompt_text):
  user_prompt_text = user_prompt_text.split()
  prompt_id = [words_to_id_ref[each] for each in user_prompt_text]
  return torch.tensor(prompt_id, dtype=torch.long, device=device)

In [36]:
X = prompt_processing(user_prompt)

In [37]:
X

tensor([10], device='mps:0')

In [38]:
X.shape

torch.Size([1])

In [39]:
model.eval()

# the code below prevent torch for tracking the gradient since we are inferring and not training
with torch.no_grad():
    prompt_logits = model(X)

In [40]:
prompt_logits.shape

torch.Size([1, 19])

In [41]:
next_token_probs = torch.softmax(prompt_logits, dim=-1)
next_token_id = torch.argmax(next_token_probs)

In [42]:
print(f"Next token is -> [{id_to_words_ref[int(next_token_id)]}]")

Next token is -> [had]


The above example just shown line by line inference using one input token. In multiple token situation, it will produce x by 19 matrix instead of 1 by 19 for single token. We need to define which prediction block to get. Please see inference 2.

### Inference 2

In [43]:
user_prompt = "mary had a"
user_prompt

'mary had a'

In [44]:
X = prompt_processing(user_prompt)

In [45]:
model.eval()

# the code below prevent torch for tracking the gradient since we are inferencing and not training
with torch.no_grad():
    prompt_logits = model(X)

prompt_logits.shape

torch.Size([3, 19])

For a string of token, we always use the last token for prediction thus we need to add a code to extract only the prediction for the last token. In this case, we are predictiing token after "a". Expected "little".


In [46]:
predict_logits = prompt_logits[-1]
predict_logits.shape

torch.Size([19])

In [47]:
next_token_probs = torch.softmax(predict_logits, dim=-1)
next_token_id = torch.argmax(next_token_probs)

In [48]:
print(f"Next token is -> [{id_to_words_ref[int(next_token_id)]}]")

Next token is -> [little]


The key difference is that we include more tokens in the user prompts. Then we need to extract the prediction of next token after the last token.

### Inference 3

In [49]:
user_prompt = "mary had a"
user_prompt

'mary had a'

In [50]:
max_token = 4

In [53]:
def prompt_inference(user_prompt, max_token, model):
  X = prompt_processing(user_prompt)
  for i in range(max_token):
    model.eval()

    # the code below prevent torch for tracking the gradient since we are inferencing and not training
    with torch.no_grad():
        prompt_logits = model(X)

    # get the last token prediction
    predict_logits = prompt_logits[-1]
    next_token_probs = torch.softmax(predict_logits, dim=-1)
    next_token_id = torch.argmax(next_token_probs)

    X = torch.cat([X, torch.tensor([next_token_id], dtype=torch.long, device=device)])
  return X


In [54]:
X = prompt_inference(user_prompt, max_token, model)
X

tensor([10,  6,  0,  9,  8,  7,  7], device='mps:0')

In [55]:
response = [id_to_words_ref[each.item()] for each in X]

In [56]:
response

['mary', 'had', 'a', 'little', 'lamb', 'its', 'its']

In [57]:
print(f"Prompt: {user_prompt}")
print(f"Response: {' '.join(response)}")

Prompt: mary had a
Response: mary had a little lamb its its


In this case, we do a inference loop. We need to set max token so that we can limit the token production. In this exercise, our total token (including the user prompt token) is more than the block size, this may produce unpredictable token once it exceed the block size. This also highlights the importance of block size a.k.a. context window.

### Inference 4

In [58]:
user_prompt = "its"
user_prompt

'its'

In [59]:
max_token = 4

In [60]:
X = prompt_inference(user_prompt, max_token, model)
X

tensor([ 7,  4, 16, 18,  2], device='mps:0')

In [61]:
response = [id_to_words_ref[each.item()] for each in X]
print(f"Prompt: {user_prompt}")
print(f"Response: {' '.join(response)}")


Prompt: its
Response: its fleece was white as


The above prompt and max token is within the block size so the prediction is good.

### Inference 5

In [62]:
user_prompt = "its"
user_prompt

'its'

In [63]:
max_token = 30

In [64]:
X = prompt_inference(user_prompt, max_token, model)
X

tensor([ 7,  4, 16, 18,  2, 11,  0,  9,  8, 10, 11,  0,  9,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1], device='mps:0')

In [65]:
response = [id_to_words_ref[each.item()] for each in X]
print(f"Prompt: {user_prompt}")
print(f"Response: {' '.join(response)}")


Prompt: its
Response: its fleece was white as snow a little lamb mary snow a little and and and and and and and and and and and and and and and and and and


What happen if our max token exceed teh vocab size. The model could not make accurate prediction after that. Please remember that the model only know the relative position within the context window a.k.a block size.

## END